In [2]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

# 1. 加载环境变量
load_dotenv()  # 加载本地的.env文件,将文件中的内容合并到环境变量


# 2. 定义工具
@tool
def get_weather(location: str) -> str:
    """
    根据位置获取当前天气情况
    :param location: 位置
    :return: 天气情况
    """
    return f"{location} 当前天气是晴天"


@tool
def square_root(x: float) -> float:
    """
    计算平方根
    :param x: 数值
    :return: 平方根
    """
    return x ** 0.5


tools = [get_weather, square_root]

# 3. 初始化模型(默认初始化)
# langchain会根据模型名称自动推断模型厂商决定base_url ,从环境变量中自动读取对应的api_key
model = init_chat_model(
    model='deepseek-v4-flash',
    temperature=1.5,
    top_p=0.75,
    # thinking={"type": "disabled"} 不是langchain支持的参数,使用 extra_body 进行传参
    # 额外参数,在跟模型交互的过程中会将额外的参数合并到请求体中一起发送给模型
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

# 4. 创建Agent(绑定模型和工具)
agent = create_agent(model=model, tools=tools)

# 5. 调用模型
response = agent.invoke({'messages': [{'role': 'user', 'content': '北京今天天气怎么样'}]})

# 6. 输出模型的返回结果
print(response)

{'messages': [HumanMessage(content='北京今天天气怎么样', additional_kwargs={}, response_metadata={}, id='3b0ae681-0fbb-42d2-ba88-9ec948dc53fb'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 350, 'total_tokens': 394, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 350}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '2423b422-4c2e-4c21-ba90-6f403f4fed26', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0464f-9092-7da1-9c89-129b7e193817-0', tool_calls=[{'name': 'get_weather', 'args': {'location': '北京'}, 'id': 'call_00_3x8YSW5ICllwgzaFgnT50882', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 350, 'o